# Motif Sample Notebook

Put the motif_sample_notebook folder in the main directory of the Motif repository.

Install the Conda environment from the main Motif directory:

```bash
conda env create -f environment.yml
```

Open the notebook in Jupyter from the motif_sample_notebook folder and select motif Conda environment. Run all code cells after another. They execute all preparation scripts and pipeline stages A to G. All generated files are saved in motif_sample_notebook.

The THYM sample contains 121 samples, 200 genes and 500 CpGs. This example uses gene–CpG pairs that were significant in an earlier analysis. It just shows how to run the pipeline.

## 1. Set up

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from IPython.display import display

assert Path.cwd().name == "motif_sample_notebook", "Run this notebook from motif_sample_notebook/."

os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
os.environ["PYTHONPATH"] = str(Path.cwd().parent)
os.environ["SNAKEMAKE_PROFILE"] = str(Path("sample_snakemake.yaml").resolve())

## 2. Prepare the cohorts data

Split the raw inputs, convert log-counts to integers and build metadata.

In [ ]:
%%bash
set -e
python -m pipeline.split_data_to_cohorts \
    --metadata data/raw/GDC-PANCAN.basic_phenotype.tsv \
    --expression data/raw/GDC-PANCAN.htseq_fpkm-uq.tsv \
    --methylation data/raw/GDC-PANCAN.methylation450.tsv \
    --raw-counts data/raw/GDC-PANCAN.htseq_counts.tsv \
    --output-dir splitted_by_project
python -m scripts.fill_cohort_dirs --config config.yaml --split-dir splitted_by_project --data-dir data
python -m scripts.repair_raw_counts data/THYM/raw_counts.tsv
python -m scripts.build_metadata --metadata data/raw/GDC-PANCAN.basic_phenotype.tsv --config config.yaml --data-directory data

## 3. Prepare references

Prepare gene and CpG annotations and the genome file for the pipeline.


In [ ]:
%%bash
set -e
python -m scripts.build_gene_annotation \
    --gencode-file resources/gencode.v22.annotation.gtf.gz \
    --hgnc-file resources/hgnc_complete_set.txt \
    --gene-annotation-output resources/gene_annotation.v22.tsv \
    --tss resources/gene_annot.tsv \
    --coding-genes resources/gencode.v22.protein_coding_gene_ids.txt
python -m scripts.build_cpg_coordinates \
    --cpg-map resources/HM450.hg38.manifest.gencode.v36.probeMap --output-file resources/cpg_annot.tsv
python -m scripts.build_cpg_annotations_for_f1 \
    --cpg-annotations-file resources/HM450.hg38.manifest.gencode.v36.zhou.tsv.gz \
    --gencode-file resources/gencode.v22.annotation.gtf.gz --output resources/cpg_gene_annotations.hg38.gencode.v22.tsv.gz
gzip -dc resources/hg38.fa.gz > resources/hg38.fa
python -m scripts.build_fasta_index resources/hg38.fa

## 4. Run Motif pipeline

In [ ]:
%%bash
set -e
python -m pipeline.run_pipeline all --executor local --cores 2

## 5. View results

In [ ]:
results = Path("results/selected_settings/rank_1/THYM")
evaluation = results / "evaluation"
pd.reset_option("display.float_format")

print("Selected MFs")
mfs = pd.read_csv(results / "selected_mfs/mfs.tsv", sep="\t")
display(mfs[["rank", "MF_ensembl", "mfs_scores"]])

print("F1 — Binding sites (unadjusted p-values)")
binding = pd.read_csv(evaluation / "binding_sites.tsv", sep="\t")
display(binding[["track", "n_mf", "p_value_sign_test"]])

print("F2 — PPI proximity (unadjusted p-values)")
proximity = pd.read_csv(evaluation / "ppi_proximity.tsv", sep="\t")
display(proximity[["measure", "n_mfs_in_network", "p_value"]])

print("F3 — Gene-set enrichment (BH-adjusted p-values)")
enrichment = pd.read_csv(evaluation / "gene_set_enrichment.tsv", sep="\t")
reported_terms = enrichment.loc[enrichment["term_has_min_mf_overlap"]]
display(reported_terms.sort_values(["adj_p_value", "p_value"])[["library_name", "term_name", "n_mf_symbols_in_term", "p_value", "adj_p_value"]])

print("F4 — Mediation (BY-adjusted joint p-values)")
triplets = pd.read_csv(evaluation / "mediation_triplets.tsv", sep="\t")
display(triplets.sort_values("joint_adj_p_value")[["MF", "CpG", "G_prime", "a_pvalue", "b_pvalue", "joint_adj_p_value"]])